In [13]:
import pyarrow as pa
import pyarrow.parquet as pq
from dbfread import DBF
import os
from itertools import islice
from pathlib import Path
import pandas as pd

In [14]:
# Convert DBF to Parquet for testing and analysis
file_path = "../dataset/CAMS AMCs NAVs 09052025/11052025150310_185847860R1.dbf"
output_dir = "../dataset/CAMS AMCs NAVs 09052025/parquet"

if os.path.exists(output_dir) is False:
    os.mkdir(output_dir)
dbf = DBF(file_path)

current_code = None
buffer = []


In [15]:
# Write the products in chunks of records to avoid memory issues
for record in islice(dbf, 1_000_000):
    code = record["PRODCODE"]

    if current_code is None:
        current_code = code

    if code != current_code:
        # write current product
        table = pa.Table.from_pylist(buffer)
        pq.write_table(
            table,
            f"{output_dir}/{current_code}.parquet"
        )

        buffer = []
        current_code = code

    buffer.append(dict(record))

In [16]:
# Write the last product if any
if buffer:
    table = pa.Table.from_pylist(buffer)
    pq.write_table(
        table,
        f"{output_dir}/{current_code}.parquet"
    )

In [17]:
# Check the number of Parquet files created and their sizes
files = list(Path(output_dir).glob("*.parquet"))

print(f"Number of Parquet files: {len(files)}")

for file in files:
    print(file.name, f"{file.stat().st_size / 1024:.2f} KB")

Number of Parquet files: 2169
B02.parquet 78.72 KB
B02D.parquet 41.11 KB
B02DZ.parquet 36.14 KB
B02G.parquet 42.27 KB
B02GK.parquet 39.81 KB
B02GZ.parquet 32.17 KB
B02Z.parquet 36.09 KB
B100.parquet 47.81 KB
B1000D.parquet 9.13 KB
B1000G.parquet 9.14 KB
B1001.parquet 28.66 KB
B1001A.parquet 7.41 KB
B1001D.parquet 6.13 KB
B1001G.parquet 6.14 KB
B1001P.parquet 12.61 KB
B1002.parquet 28.59 KB
B1002D.parquet 4.13 KB
B1002G.parquet 4.15 KB
B1003.parquet 6.09 KB
B1003A.parquet 29.26 KB
B1003D.parquet 7.56 KB
B1003P.parquet 12.52 KB
B1004.parquet 28.96 KB
B1004D.parquet 6.11 KB
B1004G.parquet 6.12 KB
B1005.parquet 50.34 KB
B1005A.parquet 28.33 KB
B1005D.parquet 4.37 KB
B1005G.parquet 4.39 KB
B1005P.parquet 34.76 KB
B1006.parquet 51.15 KB
B1006G.parquet 9.94 KB
B1007.parquet 12.27 KB
B1007D.parquet 4.39 KB
B1007G.parquet 4.41 KB
B1008.parquet 12.28 KB
B1008D.parquet 6.04 KB
B1008G.parquet 6.05 KB
B1009.parquet 13.03 KB
B1009D.parquet 7.60 KB
B1009G.parquet 7.61 KB
B100A.parquet 13.74 KB
B100B.

In [18]:
# Check the uniqueness of the columns in each Parquet file and report any errors
errors = []

for file in files:
    df = pd.read_parquet(file)

    unique_codes = df["PRODCODE"].unique()
    unique_names = df["NAME"].unique()
    unique_schemes = df["SCHEME_TYP"].unique()
    unique_isins = df["ISIN_NO"].unique()

    if (
        len(unique_codes) != 1
        or unique_codes[0] != file.stem
        or len(unique_names) != 1
        or len(unique_schemes) != 1
        or len(unique_isins) != 1
    ):
        errors.append(
            (
                file.name,
                unique_codes,
                unique_names,
                unique_schemes,
                unique_isins
            )
        )

print("Files checked:", len(files))
print("Errors:", len(errors))

if errors:
    for error in errors[:10]:
        print(error)

Files checked: 2169
Errors: 0


In [19]:
unique_counts = []

for file in files:
    df = pd.read_parquet(file)

    result = {
        "PRODCODE_FILE": file.stem,
    }

    for col in df.columns:
        result[col] = df[col].nunique(dropna=False)

    unique_counts.append(result)

unique_counts_df = pd.DataFrame(unique_counts)

display(unique_counts_df)

,PRODCODE_FILE,PRODCODE,NAME,NAV_DATE,NAV_VALUE,DIV_PERUNI,CORPDIVRAT,SCHEME_TYP,ISIN_NO,SWING_NAV
0,B02,1,1,7140,5772,1,2,1,1,2
1,B02D,1,1,3632,3077,1,1,1,1,2
2,B02DZ,1,1,3040,2856,1,1,1,1,2
3,B02G,1,1,4223,2357,1,1,1,1,2
4,B02GK,1,1,3998,2255,1,1,1,1,2
...,...,...,...,...,...,...,...,...,...,...
2164,B1391A,1,1,727,715,1,1,1,1,1
2165,B1391B,1,1,727,714,1,1,1,1,1
2166,B1391C,1,1,727,707,1,1,1,1,1
2167,B1391X,1,1,727,715,1,1,1,1,1


In [20]:
# Check the uniqueness of the columns and find any columns that have constant values across files
unique_counts_check = []

for col in unique_counts_df.columns:
    unique_values = unique_counts_df[col].unique()

    unique_counts_check.append({
        "COLUMN": col,
        "UNIQUE_COUNTS": unique_values,
        "NUMBER_OF_UNIQUE_COUNTS": len(unique_values),
        "CONSTANT": len(unique_values) == 1
    })

unique_counts_check_df = pd.DataFrame(unique_counts_check)

display(unique_counts_check_df)

,COLUMN,UNIQUE_COUNTS,NUMBER_OF_UNIQUE_COUNTS,CONSTANT
0,PRODCODE_FILE,"[B02, B02D, B02DZ, B02G, B02GK, B02GZ, B02Z, B...",2169,False
1,PRODCODE,[1],1,True
2,NAME,[1],1,True
3,NAV_DATE,"[7140, 3632, 3040, 4223, 3998, 3041, 3837, 482...",248,False
4,NAV_VALUE,"[5772, 3077, 2856, 2357, 2255, 2041, 2857, 327...",448,False
5,DIV_PERUNI,"[1, 2, 10, 4, 151]",5,False
6,CORPDIVRAT,"[2, 1, 3, 5, 4, 156]",6,False
7,SCHEME_TYP,[1],1,True
8,ISIN_NO,[1],1,True
9,SWING_NAV,"[2, 1]",2,False


In [21]:
# Clean up the output directory after checking

import shutil
shutil.rmtree(output_dir)